In [1]:
from ingestion_helpers import (
    get_connection,
    ensure_schema,
    ensure_tables,
    insert_batch,
    finalize_table,
)

TABLE_NAME = "source_enexis_test_table"

def drop_table(cursor, table_name):
    cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

example_rows = [
    {"example": "value"},
    {},  # empty row example
]

conn = get_connection()
cursor = conn.cursor()

ensure_schema(cursor)
ensure_tables(cursor, TABLE_NAME)

insert_batch(cursor, TABLE_NAME, example_rows)
finalize_table(cursor, TABLE_NAME)

conn.commit()
print("Inserted rows and finalized table.")

# Uncomment to drop the test table after running.
drop_table(cursor, TABLE_NAME)

Inserted rows and finalized table.


In [2]:
### Imports
import os
import time
import json
import logging
import shutil
import zipfile
import xml.etree.ElementTree as ET
from urllib.request import urlopen
from datetime import datetime, timedelta
#from concurrent.futures import ThreadPoolExecutor


### Custom functions
def extract_zip_recursively(folder_path, filename):
    file_path = os.path.join(folder_path, filename)

    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        filename_list = zip_ref.namelist()
        file_contains_zip = any(fname.endswith('.zip') for fname in filename_list)
        file_contains_non_zip = any(not fname.endswith('.zip') for fname in filename_list)

        if file_contains_zip:
            for fname in filename_list:
                if fname.endswith('.zip'):
                    zip_ref.extract(fname, folder_path)
                    extract_zip_recursively(folder_path, fname)

        if file_contains_non_zip:
            pass
        else:
            os.remove(file_path)

def get_file_list(folder_path, selection=None, exclusion=None):
    '''
    :param folder_path: path to folder list
    :param selection: string or list of items to include
    :param exclusion: string or list of items to exclude
    :return: list of files
    '''

    file_list = sorted(os.listdir(folder_path))

    # Convert selection to list if it's a single string
    if selection and not isinstance(selection, list):
        selection = [selection]

    # Convert exclusion to list if it's a single string
    if exclusion and not isinstance(exclusion, list):
        exclusion = [exclusion]

    # Apply selection filter if provided
    if selection:
        selection_lower = [table_name.lower() for table_name in selection]
        selected_files = [filename for filename in file_list if
                          any(table_name in filename.lower() for table_name in selection_lower)]
        file_list = selected_files

    # Apply exclusion filter if provided
    if exclusion:
        exclusion_lower = [table_name.lower() for table_name in exclusion]
        excluded_files = [filename for filename in file_list if
                          not any(excluded_item in filename.lower() for excluded_item in exclusion_lower)]
        file_list = excluded_files

    return file_list

def get_zip_file_list(folder_path, filename, selection=None, exclusion=None):

    if not filename.endswith('.zip'):
        raise(f'The file {filename} is not a .zip file')
    else:
        print(f'Retrieving .zip file list for {filename}')

    file_path = os.path.join(folder_path, filename)

    # Convert selection to list if it's a single string
    if selection and not isinstance(selection, list):
        selection = [selection]

    # Convert exclusion to list if it's a single string
    if exclusion and not isinstance(exclusion, list):
        exclusion = [exclusion]

    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()

        # Apply selection filter if provided
        if selection:
            selected_files = [filename for filename in file_list if
                              any(table_name in filename for table_name in selection)]
            file_list = selected_files

        # Apply exclusion filter if provided
        if exclusion:
            excluded_files = [filename for filename in file_list if
                              not any(excluded_item in filename for excluded_item in exclusion)]
            file_list = excluded_files

    return file_list

def read_file_from_zip(folder_path, filename, content_name, decode_as='utf-8', fallback_encoding='latin-1'):
    file_path = os.path.join(folder_path, filename)

    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        try:
            file_content_bytes = zip_ref.read(content_name)
            file_content_str = file_content_bytes.decode(decode_as)
            return file_content_str
        except UnicodeDecodeError as e:
            print(f"Error decoding with {decode_as}: {e}. Trying {fallback_encoding}...")
            try:
                file_content_str = file_content_bytes.decode(fallback_encoding)
                return file_content_str
            except UnicodeDecodeError as e:
                print(f"Error decoding with {fallback_encoding}: {e}")
                return None
            
def xml_findall_elements_dict(xml_string, element_name, namespace_uri):
    # Parse the XML string
    myroot = ET.fromstring(xml_string)

    # Register the namespace with the namespace URI
    ET.register_namespace('', namespace_uri)

    # Construct the XPath expression with the namespace prefix
    xpath_expr = f".//{{{namespace_uri}}}{element_name}"

    # Find all elements matching the XPath expression
    elements = myroot.findall(xpath_expr)

    # Convert elements to dict
    values = xml_to_dict(elements)

    return values

def xml_to_dict(element, depth=0, max_depth=50):
    if depth > max_depth:
        return None  # Stop recursion if max depth is reached

    result = {}
    for child in element:
        if child:
            child_data = xml_to_dict(child, depth + 1, max_depth)
            # Remove namespace prefix from tag name
            tag_name = child.tag.split('}')[-1]
            if tag_name in result:
                if not isinstance(result[tag_name], list):
                    result[tag_name] = [result[tag_name]]
                result[tag_name].append(child_data)
            else:
                result[tag_name] = child_data
        else:
            tag_name = child.tag.split('}')[-1]

            if child.text:
                child_data = child.text
            elif child.attrib:
                child_data = child.attrib
            else:
                child_data = None

            if tag_name in result:
                # If tag already exists, ensure it is a list
                if not isinstance(result[tag_name], list):
                    result[tag_name] = [result[tag_name]]
                result[tag_name].append(child_data)
            else:
                result[tag_name] = child_data

    return result
            

In [3]:
### Set variables
host = 'https://service.pdok.nl/kadaster/adressen/atom/v1_0/downloads/lvbag-extract-nl.zip'
selection=['.zip']
source_name = 'bag'
namespace = 'www.kadaster.nl/schemas/lvbag/imbag/objecten/v20200601'
download_folder = f"tmp/download/{source_name}"
upload_folder = f'tmp/upload/{source_name}'

In [ ]:
### Download file to temporary folder
file_name = host.split("/")[-1]

if os.path.exists(download_folder):
    shutil.rmtree(download_folder)
os.makedirs(download_folder, exist_ok=True)

# Download the zip file
file_path = os.path.join(download_folder, file_name)
with urlopen(host) as response, open(file_path, "wb") as f:
    shutil.copyfileobj(response, f)

In [2]:
### Unpack files nested structure recursively
file_list = get_file_list(download_folder, selection='.zip', exclusion=None)

# Unpack .zip files inside .zip
for filename in file_list:
    extract_zip_recursively(folder_path=download_folder, filename=filename)


NameError: name 'get_file_list' is not defined

In [ ]:
import json
import sys
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
MAX_BATCH_SIZE = 300 * 1024 * 1024  # 300 MB

start_total = datetime.now()
logging.info("Script started")

# Retrieve list of files in zipfile
exclusion = ['IO', 'GEM-WPL-RELATIE']

for object_filename in ['WPL', 'STA', 'VBO', 'PND','OPR', 'NUM']:
    
    logging.info(f"Processing object type: {object_filename}")
    
    zipfile_list = get_file_list(
        folder_path=download_folder,
        selection=object_filename,
        exclusion=exclusion
    )

    table_name = f"source_bag_{object_filename.lower()}"

    for zipfile_name in zipfile_list:
        json_set = []
        current_batch_bytes = 0

        logging.info(f"Processing zip file: {zipfile_name}")

        # Get files inside .zip file
        file_list = get_zip_file_list(
            folder_path=download_folder,
            filename=zipfile_name
        )
        
        MAX_FILES = 2  # Set to an int to limit files processed per zip
        
        for i, file_name in enumerate(file_list):
            if MAX_FILES is not None and i >= MAX_FILES:
                logging.info(f"Reached MAX_FILES={MAX_FILES}. Stopping early.")
                insert_batch(cursor, table_name, json_set)
                break
            
            is_last_file = i == len(file_list) - 1

            step_start = datetime.now()
            logging.info(f"Reading file: {file_name}")

            # Read content from .zip file
            xml_string = read_file_from_zip(
                folder_path=download_folder,
                filename=zipfile_name,
                content_name=file_name,
                decode_as='utf-8'
            )

            xml_content = ET.fromstring(xml_string)

            record_json = xml_to_dict(xml_content)['standBestand']['stand']
            current_batch_bytes += len(str(record_json).encode("utf-8"))

            json_set += record_json

            logging.info(f"File size {current_batch_bytes}")

            if current_batch_bytes >= MAX_BATCH_SIZE or is_last_file:

                insert_batch(cursor, table_name, json_set)

                json_set = []
                current_batch_bytes = 0

            elapsed = datetime.now() - step_start
            logging.info(f"Finished {file_name} in {elapsed}")


    finalize_table(cursor, table_name)
    
logging.info(f"Script finished in {datetime.now() - start_total}")

2026-03-17 16:18:16,991 | INFO | Script started
2026-03-17 16:18:16,996 | INFO | Processing object type: WPL
2026-03-17 16:18:16,999 | INFO | Found 3 zip files
2026-03-17 16:18:17,003 | INFO | Processing zip file: 9999IAWPL08032026.zip
2026-03-17 16:18:17,007 | INFO | 9999IAWPL08032026.zip contains 1 files
2026-03-17 16:18:17,008 | INFO | Reading file: 9999IAWPL08032026-000001.xml
2026-03-17 16:18:17,024 | INFO | Parsing XML
2026-03-17 16:18:17,135 | INFO | Converting XML to dict
2026-03-17 16:18:17,218 | INFO | File size 38833


Retrieving .zip file list for 9999IAWPL08032026.zip


2026-03-17 16:18:17,886 | INFO | Finished 9999IAWPL08032026-000001.xml in 0:00:00.878230
2026-03-17 16:18:17,888 | INFO | Processing zip file: 9999NBWPL08032026.zip
2026-03-17 16:18:17,891 | INFO | 9999NBWPL08032026.zip contains 1 files
2026-03-17 16:18:17,892 | INFO | Reading file: 9999NBWPL08032026-000001.xml
2026-03-17 16:18:17,904 | INFO | Parsing XML
2026-03-17 16:18:17,924 | INFO | Converting XML to dict
2026-03-17 16:18:17,940 | INFO | File size 1131016


Retrieving .zip file list for 9999NBWPL08032026.zip


2026-03-17 16:18:19,735 | INFO | Finished 9999NBWPL08032026-000001.xml in 0:00:01.842980
2026-03-17 16:18:19,737 | INFO | Processing zip file: 9999WPL08032026.zip
2026-03-17 16:18:19,743 | INFO | 9999WPL08032026.zip contains 1 files
2026-03-17 16:18:19,745 | INFO | Reading file: 9999WPL08032026-000001.xml


...

2026-03-17 16:29:41,588 | INFO | Finished 9999IANUM08032026-000001.xml in 0:00:01.127586
2026-03-17 16:29:41,589 | INFO | Processing zip file: 9999NBNUM08032026.zip
2026-03-17 16:29:41,592 | INFO | 9999NBNUM08032026.zip contains 5 files
2026-03-17 16:29:41,594 | INFO | Reading file: 9999NBNUM08032026-000001.xml
2026-03-17 16:29:41,627 | INFO | Parsing XML
2026-03-17 16:29:42,081 | INFO | Converting XML to dict


Retrieving .zip file list for 9999NBNUM08032026.zip


2026-03-17 16:29:42,449 | INFO | File size 6107509
2026-03-17 16:29:42,450 | INFO | Finished 9999NBNUM08032026-000001.xml in 0:00:00.856069
2026-03-17 16:29:42,451 | INFO | Reading file: 9999NBNUM08032026-000002.xml
2026-03-17 16:29:42,478 | INFO | Parsing XML
2026-03-17 16:29:43,260 | INFO | Converting XML to dict
2026-03-17 16:29:43,545 | INFO | File size 12274749
2026-03-17 16:29:43,546 | INFO | Finished 9999NBNUM08032026-000002.xml in 0:00:01.095447
2026-03-17 16:29:43,547 | INFO | Reached MAX_FILES=2. Stopping early.
2026-03-17 16:29:43,563 | INFO | Processing zip file: 9999NUM08032026.zip
2026-03-17 16:29:43,578 | INFO | 9999NUM08032026.zip contains 1306 files
2026-03-17 16:29:43,579 | INFO | Reading file: 9999NUM08032026-000001.xml
2026-03-17 16:29:43,633 | INFO | Parsing XML


Retrieving .zip file list for 9999NUM08032026.zip


2026-03-17 16:29:44,334 | INFO | Converting XML to dict
2026-03-17 16:29:44,644 | INFO | File size 6143675
2026-03-17 16:29:44,645 | INFO | Finished 9999NUM08032026-000001.xml in 0:00:01.066242
2026-03-17 16:29:44,646 | INFO | Reading file: 9999NUM08032026-000002.xml
2026-03-17 16:29:44,676 | INFO | Parsing XML
2026-03-17 16:29:45,292 | INFO | Converting XML to dict
2026-03-17 16:29:45,579 | INFO | File size 12012327
2026-03-17 16:29:45,580 | INFO | Finished 9999NUM08032026-000002.xml in 0:00:00.934013
2026-03-17 16:29:45,580 | INFO | Reached MAX_FILES=2. Stopping early.
2026-03-17 16:29:46,874 | INFO | Script finished in 0:11:29.883844
